# ALERT AHDC ADC — dead / bad wire analysis

This notebook analyzes the per-wire ADC values of the **AHDC** (ALERT Hyper Drift Chamber)
as a function of run number, and flags runs where a wire behaves abnormally relative to
its neighboring runs.

**Detector layout:** 8 layers with wire counts `{47, 56, 56, 72, 72, 87, 87, 99}` = **576 wires**.

**Input:** a CSV produced by `dump_alert_adc_csv.groovy` with columns

```
run, layer_number, layer_code, wire, value, graph_name
```

where `value` is the per-wire ADC integral **normalized to the trigger count** for that run
(this is the quantity plotted on the clas12mon timeline — not a raw average ADC).

The same analysis is available as a command-line script, `analyze_alert_adc.py` — see the
last section for usage examples.

## The idea in one paragraph

A wire should be judged **against itself in nearby runs**, not against a single global
number: ADC levels drift slowly across a run period (gas conditions, thresholds,
calibration), so "normal" for a wire in run 21320 is not the same as in run 23000.
We therefore build, for each wire, a *local baseline* from the adjacent runs, measure how
far each run departs from it, and flag a run if it is either (a) a sharp **outlier**
relative to that local baseline, or (b) **collapsed** relative to the wire's own overall
level. Two separate cuts are needed because each one has a blind spot the other covers.

## The two cuts

A run is flagged if **either** cut fires (`reason` records which; `low/dead` wins if both):

**Cut 1 — `outlier` (local):** &nbsp; `|robust_z| > THRESHOLD` (default 5)
Catches a run whose value departs sharply from the neighboring runs — a spike or a short dropout.

**Cut 2 — `low/dead` (global to the wire):** &nbsp; `value < DEAD_FRAC × median(value of this wire)` (default 0.2)
Catches a value that has collapsed relative to the wire's own typical level, even gradually.

**Why two cuts.** If a wire dies and *stays* dead for many runs, the rolling window inside
that stretch fills with dead values, so the local baseline itself drops to ~0 and
`robust_z` stops seeing anything unusual — the dead runs look normal *relative to their
(also-dead) neighbors*. Cut 2 compares against the wire's **overall** median instead, and
catches the sustained death. Conversely, Cut 2 alone would miss upward spikes and short
dropouts that never fall below the floor — Cut 1 gets those.

### Glossary of the variables

| variable | definition | meaning |
|---|---|---|
| `value` | — | trigger-normalized ADC for this wire in this run |
| `local_median` | rolling median over `WINDOW` runs (centered) | the level *expected* for this run given its neighbors; a median so a few dead runs in the window don't drag it down |
| `detrended` | `value − local_median` | residual after removing slow drift |
| `MAD` | `median(abs(detrended − median(detrended)))` | robust measure of normal run-to-run scatter (see below) |
| `scale` | `1.4826 × MAD` | robust standard deviation |
| `robust_z` | `detrended / scale` | how many robust sigmas this run sits from its local baseline |
| `dead_floor` | `DEAD_FRAC × median(value)` per wire | threshold for Cut 2 |
| `flag`, `reason` | OR of the cuts | which cut fired |

**Why MAD instead of the standard deviation?** The SD *squares* deviations, so a single
dead run inflates it — and the inflated spread then hides the very anomaly that caused it.
A median ignores extremes by construction: up to half the points can be anomalous before
MAD is misled.

**What is 1.4826?** A pure unit conversion. For Gaussian data, MAD converges to
`0.6745 σ` (0.6745 is the z-value of the 75th percentile — the median absolute deviation
spans the middle 50% of a normal distribution). Multiplying by `1/0.6745 = 1.4826`
rescales MAD so that on clean Gaussian data it equals the standard deviation, letting
`robust_z` be read in familiar "sigma" units.

**Worked example.** Residuals `0, 1, −1, 0.5, −0.5, 0, 1, −1, 0, 20` (one dead-run outlier):
- classical: mean = 2.0, SD ≈ 6.0 → the outlier is only **3.0σ** — *survives a 5σ cut!*
- robust: median = 0, MAD = 0.75, `scale` ≈ 1.1 → the outlier is **~18** robust sigmas — flagged decisively.

**Edge cases to know:** a perfectly flat series has `MAD = 0`, so Cut 1 is disabled and
only Cut 2 can fire. And a wire dead in *every* run has median ≈ 0, so `dead_floor` ≈ 0
and Cut 2 never fires — that case is caught by `permanently_low` in the summary (its
median compared to the detector-wide typical level across all 576 wires).

## Setup
Edit `CSV_PATH` and the three tuning knobs here.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CSV_PATH = "all.csv"   # <-- your input CSV (use test.csv for a quick run)

WINDOW    = 11    # rolling-median window in runs (local baseline)
THRESHOLD = 5.0   # |robust_z| above this  -> flag as 'outlier'
DEAD_FRAC = 0.2   # value below DEAD_FRAC x (wire median) -> flag as 'low/dead'

## 1. Load the CSV
Blank or non-numeric rows (e.g. leftover ATOF entries) are dropped automatically.

In [ ]:
def load(path):
    df = pd.read_csv(path)
    for col in ("run", "layer_number", "wire", "value"):
        df[col] = pd.to_numeric(df[col], errors="coerce")
    before = len(df)
    df = df.dropna(subset=["run", "layer_number", "wire", "value"])
    if before - len(df):
        print(f"note: dropped {before - len(df)} blank row(s)")
    for col in ("run", "layer_number", "wire"):
        df[col] = df[col].astype(int)
    df["value"] = df["value"].astype(float)
    return df.sort_values(["layer_number", "wire", "run"]).reset_index(drop=True)

df = load(CSV_PATH)
print(f"{len(df)} rows | {df.layer_number.nunique()} layers | "
      f"{df.groupby(['layer_number','wire']).ngroups} wires | "
      f"runs {df.run.min()}-{df.run.max()}")
df.head()

## 2. The detection function
This implements exactly the logic described above, per wire, in run order.
Each numbered comment matches a glossary entry.

In [ ]:
def detect_wire(g, window=WINDOW, threshold=THRESHOLD, dead_frac=DEAD_FRAC):
    """Flag anomalous runs for ONE wire's time-ordered series."""
    g = g.sort_values("run").copy()
    v = g["value"].to_numpy(float)

    # (1) local baseline: what the adjacent runs say this run *should* be
    local_med = (g["value"].rolling(window, center=True, min_periods=3)
                 .median().bfill().ffill())

    # (2) residual after removing the slow drift
    detrended = v - local_med.to_numpy()

    # (3) robust scatter: MAD, rescaled by 1.4826 to "sigma" units
    mad = np.median(np.abs(detrended - np.median(detrended)))
    scale = 1.4826 * mad if mad > 0 else 0.0          # MAD=0 -> Cut 1 disabled

    # (4) robust z-score  ->  Cut 1: |robust_z| > threshold
    robust_z = detrended / scale if scale > 0 else np.zeros_like(v)
    is_outlier = (scale > 0) & (np.abs(robust_z) > threshold)

    # (5) dead floor      ->  Cut 2: value below dead_frac x wire median
    dead_floor = dead_frac * np.median(v)
    is_low = v < dead_floor

    g["local_median"] = local_med.to_numpy()
    g["robust_z"] = robust_z
    g["dead_floor"] = dead_floor
    g["flag"] = is_outlier | is_low
    g["reason"] = np.where(is_low, "low/dead", np.where(is_outlier, "outlier", ""))
    return g

## 3. Plot one wire
Blue points = values; green dashed = local baseline; red dotted = dead floor;
red circles = flagged runs. Edit `LAYER` and `WIRE`.

In [ ]:
LAYER, WIRE = 1, 1   # <-- layer_number (1-8) and wire

def plot_wire(df, layer, wire, window=WINDOW, threshold=THRESHOLD, dead_frac=DEAD_FRAC):
    g = df[(df.layer_number == layer) & (df.wire == wire)]
    if g.empty:
        print(f"No data for layer {layer} wire {wire}"); return None
    g = detect_wire(g, window, threshold, dead_frac)
    code = g.layer_code.iloc[0] if "layer_code" in g else ""
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(g.run, g.value, "-", color="0.7", lw=1)
    ax.scatter(g.run, g.value, s=18, color="steelblue", label="value")
    ax.plot(g.run, g.local_median, "--", color="green", lw=1.2,
            label=f"local median (window={window})")
    ax.axhline(g.dead_floor.iloc[0], color="red", ls=":", lw=1,
               label=f"dead floor ({dead_frac:g}x median)")
    fl = g[g.flag]
    if not fl.empty:
        ax.scatter(fl.run, fl.value, s=70, facecolors="none",
                   edgecolors="red", linewidths=1.7, label="flagged")
    ax.set_xlabel("run number"); ax.set_ylabel("AHDC ADC value")
    ax.set_title(f"AHDC ADC - layer {layer} (code {code}), wire {wire}")
    ax.legend(fontsize=8); plt.show()
    return g

g = plot_wire(df, LAYER, WIRE)
if g is not None and g.flag.any():
    display(g[g.flag][["run", "value", "local_median", "robust_z", "reason"]])

## 4. Scan every wire
Applies the two cuts to all wires and builds a per-wire summary:

- **`frac_flagged`** = flagged runs / total runs (ranking only — no automatic cut on it)
- **`permanently_low`** = wire median below `DEAD_FRAC ×` the detector-wide typical level
  (catches wires dead in *every* run, which the per-run cuts structurally cannot flag)

The tool ranks suspects; the final "this wire is dead" verdict is the analyst's call.
Expect a few wires flagged for one isolated run (statistical noise) — the wires that
matter have a high `frac_flagged` or `permanently_low = True`.

In [ ]:
res = pd.concat([detect_wire(g) for _, g in df.groupby(["layer_number", "wire"])],
                ignore_index=True)
flagged = res[res.flag].copy()
print(f"{len(flagged)} flagged (run,wire) entries out of {len(res)} "
      f"({100*len(flagged)/len(res):.2f}%)")

global_typical = res.groupby(["layer_number","wire"]).value.median().median()
summary = (res.groupby(["layer_number","wire"])
           .agg(n_runs=("run","size"), n_flagged=("flag","sum"),
                median_value=("value","median")).reset_index())
summary["frac_flagged"] = summary.n_flagged / summary.n_runs
summary["permanently_low"] = summary.median_value < DEAD_FRAC * global_typical

summary.sort_values(["permanently_low","frac_flagged"], ascending=False).head(20)

## 5. Save the flagged runs

In [ ]:
cols = ["run","layer_number","layer_code","wire","value",
        "local_median","robust_z","reason"]
flagged[[col for col in cols if col in flagged.columns]].to_csv("flagged.csv", index=False)
print(f"wrote flagged.csv ({len(flagged)} rows)")

## 6. Visual check of the worst offenders
Plots the top suspects from the summary.

In [ ]:
worst = summary.sort_values(["permanently_low","frac_flagged"],
                            ascending=False).head(5)
for _, row in worst.iterrows():
    plot_wire(df, int(row.layer_number), int(row.wire))

## Command-line equivalent (`analyze_alert_adc.py`)

The same analysis runs from a terminal — useful for batch work or on ifarm:

```bash
# scan every wire, write flagged runs + print the per-wire summary
python analyze_alert_adc.py all.csv --scan flagged.csv

# plot one wire and list its flagged runs
python analyze_alert_adc.py all.csv --layer 1 --wire 1 --plot l1_w1.png

# custom tuning (same three knobs as this notebook)
python analyze_alert_adc.py all.csv --scan flagged.csv \
    --window 15 --threshold 4 --dead-frac 0.3
```

| option | default | meaning |
|---|---|---|
| `--layer N --wire M` | – | which wire to plot |
| `--plot FILE.png` | auto | output plot path |
| `--scan [FILE.csv]` | `alert_adc_flagged.csv` | scan all wires, write flagged runs |
| `--window N` | 11 | rolling-median window (local baseline) |
| `--threshold X` | 5.0 | robust-z cutoff for Cut 1 |
| `--dead-frac F` | 0.2 | dead-floor fraction for Cut 2 |

**Tuning guidance:** widen `--window` for long stable run periods, narrow it if the
baseline drifts fast; lower `--threshold` to catch subtler dropouts (more sensitivity,
more false flags); raise `--dead-frac` toward 0.5 if "dead" should mean "clearly below
normal" rather than "near zero". Record the exact parameters used when you save a
`flagged.csv` — the flag counts depend strongly on all three.